# Module 1 · Lesson 06: Streaming Responses

**Streaming** delivers tokens one-by-one as they are generated, instead of waiting
for the full response. This is essential for building responsive AI applications.

## What you will learn
1. Basic streaming with OpenAI
2. Measuring **Time to First Token (TTFT)** and throughput
3. Streaming with Anthropic (different pattern!)
4. Streaming vs blocking — perceived latency
5. When to use streaming vs blocking

---
> ⚡ **Why streaming matters:** A 2-second delay feels slow. But seeing tokens
> appear instantly (even if the full response takes 2 seconds) feels fast and engaging.

In [1]:
import os, time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown
 
load_dotenv(Path.cwd().parent / ".env")
 
from openai import OpenAI
client = OpenAI()
print("✅ Client ready")

✅ Client ready


---
## 1. Basic Streaming (OpenAI)

Set `stream=True` and iterate over the response. Each chunk contains a small piece of text.

In [2]:
print("🔄 Streaming response:\n")
 
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a short poem about Python programming."}],
    max_tokens=150,
    stream=True     # ← This enables streaming!
)
 
full_text = ""
for chunk in stream:
    # Each chunk may have content or be empty (keep-alive)
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)
        full_text += delta
 
print(f"\n\n📊 Total length: {len(full_text)} characters")

🔄 Streaming response:

In the realm of code, a serpent glides,  
With elegance and grace, where magic abides.  
From simple scripts to complex designs,  
In Python's embrace, creativity shines.  

Indentations guide, like a gentle hand,  
Logic flows freely, like grains of sand.  
Libraries vast, a treasure trove wide,  
In this digital world, let curiosity be your guide.  

With each line we craft, solutions unfold,  
In variables and loops, new stories are told.  
So here’s to the coder, both novice and sage,  
In the dance of Python, we all find our stage.  

📊 Total length: 544 characters


> 💡 **Key Pattern:** `chunk.choices[0].delta.content` — note it's `delta` not `message`!
> The delta contains only the *new* content in each chunk.

---
## 2. Timing — TTFT and Throughput

Two critical metrics for streaming:
- **TTFT** (Time to First Token): How quickly the first token arrives
- **Throughput**: Tokens per second after streaming starts

In [3]:
# ── Example 2: Timing metrics ─────────────────────────
prompt = "Explain what machine learning is in 3 sentences."

start = time.perf_counter()
ttft = None
token_count = 0
full_text = ""

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    stream=True
)

print("🔄 Streaming with timing:\n")

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000  # ms
        print(delta, end="", flush=True)
        full_text += delta
        token_count += 1

total_ms = (time.perf_counter() - start) * 1000
tps = token_count / (total_ms / 1000) if total_ms > 0 else 0

print(f"\n\n{'─'*40}")
print(f"⏱  Time to First Token (TTFT): {ttft:.0f} ms")
print(f"⏱  Total time:                 {total_ms:.0f} ms")
print(f"📊 Tokens received:             {token_count}")
print(f"⚡ Throughput:                  {tps:.1f} tokens/sec")

🔄 Streaming with timing:

Machine learning is a subset of artificial intelligence that enables computers to learn from and make predictions based on data without being explicitly programmed. It involves the use of algorithms that can identify patterns and relationships within large datasets, allowing the system to improve its performance over time as it is exposed to more data. Machine learning is widely applied in various fields, including finance, healthcare, and technology, to automate tasks, enhance decision-making, and provide insights.

────────────────────────────────────────
⏱  Time to First Token (TTFT): 2790 ms
⏱  Total time:                 4143 ms
📊 Tokens received:             87
⚡ Throughput:                  21.0 tokens/sec


---
## 3. Streaming with Anthropic

Anthropic uses a **context manager** pattern instead of `stream=True`:

In [4]:
# ── Example 3: Anthropic streaming ────────────────────
try:
    import anthropic
    ac = anthropic.Anthropic()

    print("🔄 Claude streaming:\n")

    with ac.messages.stream(
        model="claude-sonnet-4-6",
        max_tokens=150,
        messages=[{"role": "user", "content": "Write a haiku about AI."}]
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)

    print("\n\n✅ Anthropic streaming works!")

except Exception as e:
    print(f"⚠️ Anthropic not available: {e}")
    print("   That's OK — OpenAI streaming is the primary focus.")

🔄 Claude streaming:

Here's a haiku about AI:

*Silent circuits think*
*Patterns woven from our words*
*Who taught whom to dream?*

✅ Anthropic streaming works!


### API Comparison: Streaming

```python
# OpenAI streaming
stream = client.chat.completions.create(
    ..., stream=True
)
for chunk in stream:
    text = chunk.choices[0].delta.content

# Anthropic streaming
with client.messages.stream(...) as stream:
    for text in stream.text_stream:
        print(text)
```

---
## 4. Streaming vs Blocking — Latency Comparison

In [5]:
# ── Example 4: Streaming vs Blocking ──────────────────
prompt = "List 5 benefits of learning Python."

# Method 1: Blocking (wait for full response)
start = time.perf_counter()
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200
)
blocking_time = (time.perf_counter() - start) * 1000

# Method 2: Streaming (measure TTFT)
start = time.perf_counter()
stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}],
    max_tokens=200,
    stream=True
)
streaming_ttft = None
for chunk in stream:
    if chunk.choices[0].delta.content and streaming_ttft is None:
        streaming_ttft = (time.perf_counter() - start) * 1000
streaming_total = (time.perf_counter() - start) * 1000

# Results
print("⏱  Perceived Latency Comparison")
print(f"{'─'*45}")
print(f"Blocking:  User waits {blocking_time:.0f} ms to see ANYTHING")
print(f"Streaming: User sees first token in {streaming_ttft:.0f} ms")
print(f"           (Full response in {streaming_total:.0f} ms)")
print(f"\n💡 Perceived improvement: {blocking_time - streaming_ttft:.0f} ms faster!")

⏱  Perceived Latency Comparison
─────────────────────────────────────────────
Blocking:  User waits 4829 ms to see ANYTHING
Streaming: User sees first token in 1209 ms
           (Full response in 5081 ms)

💡 Perceived improvement: 3619 ms faster!


---
## 5. Real-World Exercise: Scrape & Stream

Let's combine streaming with a practical use case: **scrape a webpage and stream
a summary of its content**. This is inspired by community projects that scrape
news/JIRA/email and summarize with LLMs.

```
Web Page --> requests + BeautifulSoup --> Raw Text --> GPT (stream) --> Summary
```

In [7]:
import requests
from bs4 import BeautifulSoup

def scape_text(url: str, max_chars: int = 3000) -> str:
    """Fetch a URL and exteract clean content"""
    headers = {"User-Agent": "Mozilla/5.0 (Educational Bot)"}
    r = requests.get(url, headers=headers, timeout=10)
    soup = BeautifulSoup(r.text, "html.parser")

    # Remove noise
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    text = soup.get_text(separator="\n", strip=True)
    return text[:max_chars]

try:
    page_text = scape_text("https://diaviou.aueb.gr/programs/36148-AI-for-Developers--Design,-Build,-Deploy-LLM-powered-Applications-1001200")
    print(f"✅ Scraped {len(page_text)} characters")
    print(f"Preview:\n{page_text[:200]}...")
except:
    # Fallback if no internet connection...
    page_text = """Python is a programming language that lets you work quickly
and integrate systems more effectively. Python is powerful and fast, plays well
with others, runs everywhere, is friendly and easy to learn, and is open.
Python can be used for web development, data analysis, machine learning,
scientific computing, automation, and more."""
    print(f"Using fallback text ({len(page_text)} chars): {e}")
    pass

✅ Scraped 3000 characters
Preview:
AI for Developers: Design, Build, Deploy LLM-powered Applications
210 8203913
secretariat@diaviou.aueb.gr
ΠΡΟΓΡΑΜΜΑΤΑ
Αρχική
ΠΡΟΓΡΑΜΜΑΤΑ
ΠΡΟΓΡΑΜΜΑΤΑ
Εκτύπωση
ΠΡΟΓΡΑΜΜΑΤΑ
AI for Developers: Design, Bui...


In [8]:
print("Streaming summary of scraped content:\n")
print("=" * 50)

start = time.perf_counter()
ttft = None
full_summary = ""
md_display = display(Markdown(""), display_id=True)

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "Summarize web page content in 3-5 bullet points."
                                      "Be concise and highlight key facts."},
        {"role": "user", "content": f"Summarize this web page content: \n\n {page_text}"}
    ],
    max_tokens=500,
    stream=True
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        if ttft is None:
            ttft = (time.perf_counter() - start) * 1000
            md_display.update(Markdown(full_summary))
        full_summary += delta
        md_display.update(Markdown(full_summary))

total_ms = (time.perf_counter() - start) * 1000

print(f"\n\n {"=" * 50}")
print(f"TTFT: {ttft:.0f} ms | Total: {total_ms:.0f} ms")
print(f"Input: {len(page_text)} chars scaped | Output: {len(full_summary)} chars summary")


Streaming summary of scraped content:



- The program focuses on AI applications, specifically Large Language Models (LLMs), and teaches participants how to integrate these models into applications using Python and FastAPI.
- Training includes practical workshops, prompt engineering techniques, development of Retrieval-Augmented Generation (RAG) solutions, and creation of autonomous AI agents.
- Participants can attend either in-person, with limited seating based on registration, or through live streaming, which allows for flexible learning and access to recorded sessions.
- Each participant will receive free educational materials and is expected to complete a final project that showcases architectural, development, and creative skills.
- The program ensures a comprehensive understanding of AI fundamentals, LLMs, and practical implementation skills for real-world applications.



TTFT: 2032 ms | Total: 3445 ms
Input: 3000 chars scaped | Output: 849 chars summary


> **Exercise:** Modify the scraper to fetch a news article of your choice.
> Compare the streaming summary with a blocking call — which feels faster to the user?
> Try adding `temperature=0` for factual summaries vs `temperature=0.7` for creative ones.

---
## When to Use Each Pattern

| Pattern | Use When |
|---------|----------|
| **Streaming** | Chat interfaces, long responses, user-facing apps |
| **Blocking** | Background processing, batch jobs, structured output (JSON) |

---
## Key Takeaways 📝

| Concept | Detail |
|---------|--------|
| **TTFT** | Time to first token — the key UX metric |
| **Throughput** | Tokens/second — depends on model and load |
| **OpenAI** | `stream=True` + iterate chunks |
| **Anthropic** | `messages.stream()` context manager |
| **delta vs message** | Streaming gives `.delta.content`, blocking gives `.message.content` |
| **Scrape + Stream** | Combine web scraping with streaming for real-time summaries |

---
**Next:** `07_response_inspector.ipynb` — Understand the full response metadata